In [1]:
# importando bibliotecas utilizadas
# importing used libraries
import pandas as pd
from sklearn import tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

In [2]:
# lendo os dados
# reading the data
treino = pd.read_csv("DATA/train.csv")
test = pd.read_csv("DATA/test.csv")

Visualizando a base de treino e test 

Visualizing the training and test sets

In [3]:
treino.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [4]:
test.head(3)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q


Verificando as informações da base 

Verifying database information

In [5]:
treino.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [6]:
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    str    
 3   Sex          418 non-null    str    
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    str    
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     str    
 10  Embarked     418 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 36.1 KB


Verificando a cardinalidade dos dados 

Checking data cardinality

In [7]:
treino.nunique().sort_values(ascending=False)

PassengerId    891
Name           891
Ticket         681
Fare           248
Cabin          147
Age             88
SibSp            7
Parch            7
Embarked         3
Pclass           3
Survived         2
Sex              2
dtype: int64

In [8]:
test.nunique().sort_values(ascending=False)

PassengerId    418
Name           418
Ticket         363
Fare           169
Age             79
Cabin           76
Parch            8
SibSp            7
Pclass           3
Embarked         3
Sex              2
dtype: int64

Verificando os valores nulos 

checking for null values

In [9]:
treino.isnull().sum().sort_values(ascending=False).head(5)

Cabin          687
Age            177
Embarked         2
PassengerId      0
Name             0
dtype: int64

In [10]:
test.isnull().sum().sort_values(ascending=False).head(5)

Cabin     327
Age        86
Fare        1
Name        0
Pclass      0
dtype: int64

Temos colunas que possuem valores vazios na base de teste que não estão vazias na base de treino (Nesse caso, precisaremos tratar essas colunas apenas na base de teste)

We have columns containing empty values ​​in the test set that are not empty in the training set (in this case, we will need to handle these columns only in the test set).

Fazendo o tratamento de dados para valores nulos e cardinalidades

Handling null values ​​and cardinalities during data processing

In [11]:
treino = treino.drop(['Name', 'Ticket', 'Cabin'], axis=1)

In [12]:
test = test.drop(['Name', 'Ticket', 'Cabin'],axis=1)

Para lidar com valores nulos no campo "idade", está sendo utilizada a média de idade.

To handle null values ​​in the "age" field, the mean age is being used.

In [13]:
treino.Age.mean()

np.float64(29.69911764705882)

In [14]:
treino.loc[treino.Age.isnull(),'Age'] = treino.Age.mean()

In [15]:
test.loc[test.Age.isnull(),'Age'] = test.Age.mean()

Como ainda existem duas colunas com valores nulos — "Embarked" e "Fare" —, a moda será utilizada para preencher os valores ausentes.

Since there are still two columns with null values—"Embarked" and "Fare"—the mode will be used to fill in the missing values.

In [16]:
treino.Embarked.mode()[0]

'S'

In [17]:
treino.loc[treino.Embarked.isnull(),'Embarked'] = treino.Embarked.mode()[0]

In [18]:
test.loc[test.Fare.isnull(),'Fare'] = test.Fare.mean()

Confirmando se ainda existe valores nulos

Verifying if there are still null values.

In [19]:
treino.isnull().sum().sort_values(ascending=False).head(5)

PassengerId    0
Survived       0
Pclass         0
Sex            0
Age            0
dtype: int64

In [20]:
test.isnull().sum().sort_values(ascending=False).head(5)

PassengerId    0
Pclass         0
Sex            0
Age            0
SibSp          0
dtype: int64

Vamos considerar as colunas que não são de texto para o nosso banco de dados.

Let's consider the non-text columns for our database.

In [21]:
col_treino_nr = treino.columns[treino.dtypes != 'object']
col_treino_nr

Index(['PassengerId', 'Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch',
       'Fare', 'Embarked'],
      dtype='str')

As colunas 'Sex' e 'Embarked' são removidas por serem strings, restando apenas valores numéricos.

The 'Sex' and 'Embarked' columns are removed because they are strings, leaving only numerical values.

In [23]:
col_treino_nr = col_treino_nr.drop(['Sex', 'Embarked'])

In [24]:
treino_nr = treino.loc[:, col_treino_nr]
treino_nr

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
0,1,0,3,22.000000,1,0,7.2500
1,2,1,1,38.000000,1,0,71.2833
2,3,1,3,26.000000,0,0,7.9250
3,4,1,1,35.000000,1,0,53.1000
4,5,0,3,35.000000,0,0,8.0500
...,...,...,...,...,...,...,...
886,887,0,2,27.000000,0,0,13.0000
887,888,1,1,19.000000,0,0,30.0000
888,889,0,3,29.699118,1,2,23.4500
889,890,1,1,26.000000,0,0,30.0000


In [25]:
col_test_nr = test.columns[test.dtypes != 'object']
col_test_nr

Index(['PassengerId', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked'],
      dtype='str')

In [26]:
col_test_nr = col_test_nr.drop(['Sex', 'Embarked'])

In [27]:
teste_nr = test.loc[:, col_test_nr]
teste_nr

,PassengerId,Pclass,Age,SibSp,Parch,Fare
0,892,3,34.50000,0,0,7.8292
1,893,3,47.00000,1,0,7.0000
2,894,2,62.00000,0,0,9.6875
3,895,3,27.00000,0,0,8.6625
4,896,3,22.00000,1,1,12.2875
...,...,...,...,...,...,...
413,1305,3,30.27259,0,0,8.0500
414,1306,1,39.00000,0,0,108.9000
415,1307,3,38.50000,0,0,7.2500
416,1308,3,30.27259,0,0,8.0500


Antes de utilizar os algoritmos, precisamos dividir o conjunto de dados de treinamento em conjuntos de TREINAMENTO e VALIDAÇÃO; para isso, utilizaremos o "train_test_split".

Before using the algorithms, we need to split the training dataset into TRAINING and VALIDATION sets; to do this, we will use "train_test_split".

In [28]:
X = treino_nr.drop(['PassengerId', 'Survived'],axis= 1)
Y = treino.Survived

In [29]:
X_treino, X_Val, Y_treino, Y_Val = train_test_split(X, Y, test_size=0.33, random_state=42)

Realizado a separação, agora podemos selecionar um modelo para classificar esses dados ('Arvore de classificação', 'classificação dos visinhos proximos', regressão logistica')

Now that the data has been split, we can select a model to classify it (e.g., 'classification tree', 'k-nearest neighbors', 'logistic regression').

para ARVORE DE CLASSIFICAÇÃO

for CLASSIFICATION TREE

In [30]:
clf_ac = tree.DecisionTreeClassifier(random_state=42)

In [ ]:
clf_ac = clf_ac.fit(X_treino, Y_treino)

In [32]:
y_pred_ac = clf_ac.predict(X_Val)

In [33]:
print(X_treino.dtypes)

Pclass      int64
Age       float64
SibSp       int64
Parch       int64
Fare      float64
dtype: object


para CLASSIFICAÇÃO DOS VISINHOS PROXIMOS

for NEAREST NEIGHBOR CLASSIFICATION

In [34]:
clf_knn = KNeighborsClassifier(n_neighbors=3)

In [35]:
clf_knn = clf_knn.fit(X_treino,Y_treino)

In [36]:
y_pred_knn = clf_knn.predict(X_Val)

para REGRESSÃO LOGISTICA

for LOGISTIC REGRESSION

In [37]:
clf_rl = LogisticRegression(random_state=42, max_iter=1000)

In [38]:
clf_rl = clf_rl.fit(X_treino,Y_treino)

In [39]:
y_pred_rl = clf_rl.predict(X_Val)

Uma vez realizados os três métodos, podemos determinar qual deles apresenta a maior precisão.

Once all three methods have been carried out, we can determine which one has the highest accuracy.

In [40]:
accuracy_score(Y_Val, y_pred_ac)

0.6169491525423729

In [41]:
accuracy_score(Y_Val, y_pred_knn)

0.6542372881355932

In [42]:
accuracy_score(Y_Val, y_pred_rl)

0.7254237288135593

avaliando a matriz de confusão

evaluating the confusion matrix

In [43]:
confusion_matrix(Y_Val, y_pred_ac)

array([[125,  50],
       [ 63,  57]])

In [44]:
confusion_matrix(Y_Val, y_pred_knn)

array([[133,  42],
       [ 60,  60]])

In [45]:
confusion_matrix(Y_Val, y_pred_rl)

array([[156,  19],
       [ 62,  58]])

Após a avaliação, constatamos que a regressão logística oferece a maior precisão; portanto, iremos utilizá-la para a nossa previsão.

After the evaluation, we found that logistic regression offers the highest precision, so we will use it for our prediction.

Verificando a base de treino e teste

Checking the training and test sets

In [46]:
X_treino.head(3)

,Pclass,Age,SibSp,Parch,Fare
6,1,54.000000,0,0,51.8625
718,3,29.699118,0,0,15.5000
685,2,25.000000,1,2,41.5792


In [47]:
teste_nr.head(3)

,PassengerId,Pclass,Age,SibSp,Parch,Fare
0,892,3,34.5,0,0,7.8292
1,893,3,47.0,1,0,7.0000
2,894,2,62.0,0,0,9.6875


Como o conjunto de teste contém 'PassengerId', mas o conjunto de treinamento não, vamos removê-lo para realizar a análise e, posteriormente, adicioná-lo novamente para criar o arquivo de submissão para o Kaggle.

Since the test set contains 'PassengerId' but the training set does not, we will remove it to perform the analysis, and then add it back later to create the submission file for Kaggle.

In [48]:
x_teste = teste_nr.drop("PassengerId", axis=1)

In [49]:
y_pred = clf_rl.predict(x_teste)

In [50]:
teste_nr['Survived'] = y_pred

In [51]:
base_envio = teste_nr[['PassengerId', 'Survived']]

Base de envio

Shipping base

In [54]:
base_envio.to_csv('resultados.csv', index=False)